# RAG System Demo
## Financial Document Question Answering (Apple & Tesla 10-K)

This notebook demonstrates the complete RAG pipeline for answering questions from 10-K filings.

**Configuration:**
- **LLM**: OpenAI gpt-4o-mini (via API)
- **Embeddings**: sentence-transformers/all-MiniLM-L6-v2
- **Vector Store**: FAISS (CPU)
- **Documents**: Apple 10-K (FY 2024), Tesla 10-K (FY 2023)

**Model Loading Efficiency:**
- The embedding model is loaded **ONCE** when the pipeline initializes
- OpenAI is API-based (no local model loading)
- Multiple questions can be asked without re-initialization ✅

### 1. Setup and Installation

In [ ]:
import sys
sys.path.append('..')

import time
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables (API key from .env)
load_dotenv()

print("="*60)
print("RAG FOR APPLE & TESLA 10-K FILINGS")
print("="*60)
print("LLM: OpenAI gpt-4o-mini")
print("Embeddings: sentence-transformers/all-MiniLM-L6-v2")
print("Vector Store: FAISS (CPU)")
print("="*60)

### 2. Initialize RAG Pipeline

The pipeline initializes **once** and loads:
- Embedding model (sentence-transformers)
- FAISS vector index
- OpenAI client

Subsequent questions reuse these components (efficient!).

In [ ]:
from src.config import RAGConfig
from src.pipeline.rag_pipeline import RAGPipeline

print("Initializing RAG Pipeline...")
print("  - Loading embedding model (sentence-transformers)")
print("  - Creating OpenAI client")
print("  - Building FAISS index")

# Initialize pipeline (models loaded ONCE here)
config = RAGConfig()
pipeline = RAGPipeline(config)

# Index documents
pdf_paths = list(Path("../data/pdfs").glob("*.pdf"))
print(f"\nFound {len(pdf_paths)} PDF files:")
for pdf in pdf_paths:
    print(f"  - {pdf.name}")

print("\nIndexing documents...")
pipeline.index_documents(pdf_paths)

print("\n✅ Pipeline ready! Models loaded once - reused for all questions.")

### 3. Ground Truth Evaluation Questions

The following 13 questions are used for evaluation. Expected answers are shown as comments.

**Apple Questions (Q1-Q5)** - Source: Apple 10-K FY 2024
**Tesla Questions (Q6-Q10)** - Source: Tesla 10-K FY 2023
**Out-of-Scope Questions (Q11-Q13)** - Should refuse to answer

#### Apple Questions (Q1-Q5)

In [ ]:
# Q1: Apple's total revenue FY 2024
# Expected Answer: $391,036 million
# Source: Apple 10-K, Item 8, p. 28

result = pipeline.answer_question("What was Apple's total revenue for fiscal year 2024?")
print("Q1: Apple's total revenue FY 2024")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q2: Apple's common stock outstanding (Oct 18, 2024)
# Expected Answer: 15,115,823,000 shares
# Source: Apple 10-K, first paragraph

result = pipeline.answer_question("How many shares of common stock did Apple have outstanding as of October 18, 2024?")
print("Q2: Apple's common stock outstanding")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q3: Apple's total term debt (Sept 28, 2024)
# Expected Answer: $96,662 million
# Source: Apple 10-K, Item 8, Note 9, p. 39

result = pipeline.answer_question("What was Apple's total term debt as of September 28, 2024?")
print("Q3: Apple's total term debt")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q4: Filing date of Apple's 2024 10-K
# Expected Answer: November 1, 2024
# Source: Apple 10-K, Signature page

result = pipeline.answer_question("What was the filing date of Apple's 2024 10-K?")
print("Q4: Apple's 10-K filing date")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q5: Unresolved SEC staff comments?
# Expected Answer: No
# Source: Apple 10-K, Item 1B

result = pipeline.answer_question("Does Apple have any unresolved SEC staff comments?")
print("Q5: Unresolved SEC staff comments")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

#### Tesla Questions (Q6-Q10)

In [ ]:
# Q6: Tesla's total revenue FY 2023
# Expected Answer: $96,773 million
# Source: Tesla 10-K, Item 8

result = pipeline.answer_question("What was Tesla's total revenue for fiscal year 2023?")
print("Q6: Tesla's total revenue FY 2023")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q7: Tesla revenue from Automotive Sales
# Expected Answer: ~84%
# Source: Tesla 10-K, Item 7

result = pipeline.answer_question("What percentage of Tesla's revenue comes from Automotive Sales?")
print("Q7: Tesla's Automotive Sales percentage")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q8: Tesla's dependency on Elon Musk
# Expected Answer: Central to strategy, loss could disrupt operations
# Source: Tesla 10-K, Risk Factors

result = pipeline.answer_question("What are the risks associated with Tesla's dependency on Elon Musk?")
print("Q8: Tesla's dependency on Elon Musk")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q9: Tesla's current vehicles
# Expected Answer: Model S, 3, X, Y, Cybertruck
# Source: Tesla 10-K, Item 1

result = pipeline.answer_question("What types of vehicles does Tesla currently produce and deliver?")
print("Q9: Tesla's current vehicle lineup")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

In [ ]:
# Q10: Tesla's lease pass-through arrangements
# Expected Answer: Finance solar systems with investors; customers sign PPAs
# Source: Tesla 10-K, Item 7

result = pipeline.answer_question("What are Tesla's lease pass-through arrangements?")
print("Q10: Tesla's lease pass-through arrangements")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

#### Out-of-Scope Questions (Q11-Q13)

These questions should be **refused** by the RAG system (hallucination prevention).

In [ ]:
# Q11: Tesla's stock price forecast 2025
# Expected Answer: NOT ANSWERABLE (out of scope)
# The RAG should refuse to answer this question

result = pipeline.answer_question("What is Tesla's stock price forecast for 2025?")
print("Q11: Tesla's stock price forecast 2025 (OUT OF SCOPE)")
print(f"Answer: {result['answer']}")
print("Should refuse - stock forecasts not in 10-K filings")

In [ ]:
# Q12: Apple CFO (2025)
# Expected Answer: NOT ANSWERABLE (out of scope)
# The RAG should refuse - 2025 info not in documents

result = pipeline.answer_question("Who is Apple's CFO in 2025?")
print("Q12: Apple CFO 2025 (OUT OF SCOPE)")
print(f"Answer: {result['answer']}")
print("Should refuse - 2025 data not in 10-K documents")

In [ ]:
# Q13: Tesla HQ color
# Expected Answer: NOT ANSWERABLE (out of scope)
# The RAG should refuse - irrelevant to 10-K content

result = pipeline.answer_question("What color is Tesla's headquarters building?")
print("Q13: Tesla HQ color (OUT OF SCOPE)")
print(f"Answer: {result['answer']}")
print("Should refuse - not in 10-K filings")

### 4. Interactive Mode

In [ ]:
# Interactive question answering
def ask_interactive():
    """Interactive Q&A loop."""
    print("\n" + "="*60)
    print("INTERACTIVE MODE")
    print("="*60)
    print("Ask questions about Apple or Tesla 10-K filings.")
    print("Type 'quit' to exit.\n")
    
    while True:
        question = input("\nYour question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye!")
            break
        
        if not question:
            continue
        
        print("\n⏱️  Processing...\n")
        start_time = time.time()
        result = pipeline.answer_question(question)
        elapsed = time.time() - start_time
        
        print("="*60)
        print(f"Answer: {result['answer']}")
        print(f"\nSources: {result['sources']}")
        print(f"\nTime: {elapsed:.2f}s")
        print("="*60)

# Uncomment to run interactive mode
# ask_interactive()

In [ ]:
# Or ask a single custom question
custom_question = "Your question here"

# result = pipeline.answer_question(custom_question)
# print(f"Question: {custom_question}")
# print(f"\nAnswer: {result['answer']}")
# print(f"\nSources: {result['sources']}")

### 5. Pipeline Statistics

View information about the RAG pipeline.

In [ ]:
# Get pipeline statistics
stats = pipeline.get_pipeline_stats()

print("="*60)
print("PIPELINE STATISTICS")
print("="*60)

print(f"\nIndexed: {stats['is_indexed']}")
print(f"\nConfiguration:")
for key, value in stats['config'].items():
    print(f"  {key}: {value}")

if 'vector_store' in stats:
    print(f"\nVector Store:")
    for key, value in stats['vector_store'].items():
        print(f"  {key}: {value}")

print("="*60)